In [1]:
import pandas as pd
from sqlalchemy import text

# Utils moduli
from utils.data_loader import load_all_orders, load_returns, load_users
from utils.data_cleaner import clean_orders
from utils.db_connector import get_engine, create_database, test_connection
from utils.Exporter import export_csv

# Putanje do foldera
RAW_DIR       = 'data/raw/'
PROCESSED_DIR = 'data/processed/'

print("✅ Import uspešan!")

✅ Import uspešan!


In [2]:
create_database ('smart_store')

engine = get_engine('smart_store')

test_connection(engine)

Baza 'smart_store' kreirana (ili vec postoji)
Konekcija kreirana: localhost/smart_store
Konekcija uspesna!


True

In [3]:
orders_raw = load_all_orders(RAW_DIR)

returns_raw = load_returns(RAW_DIR + 'Returns.csv')

users_raw = load_users(RAW_DIR + 'Users.csv')

Ucitan Orders_East.csv: 474 redova
Ucitan Orders_West.csv: 470 redova
Ucitan Orders_South.csv: 441 redova
Ucitan Orders_Central.csv: 566 redova
Ukupno redova nakon spajanja: 1951
Ucitan Returns: 300 redova
Ucitan Users: 4 redova


In [4]:
orders_raw.head()

,Row ID,Order Priority,Discount,Unit Price,Shipping Cost,Customer ID,Customer Name,Ship Mode,Customer Segment,Product Category,...,Region,State or Province,City,Postal Code,Order Date,Ship Date,Profit,Quantity ordered new,Sales,Order ID
0,21776,Critical,0.06,9.48,7.29,11,Marcus Dunlap,Regular Air,Home Office,Furniture,...,East,New Jersey,Roselle,7203,2015-02-15,2015-02-17,-53.8096,22,211.15,90192
1,18181,Critical,0.00,4.42,4.99,15,Timothy Reese,Regular Air,Small Business,Office Supplies,...,East,New York,Smithtown,11787,2015-04-08,2015-04-09,-59.8200,7,33.47,86837
2,20925,Medium,0.01,35.94,6.66,15,Timothy Reese,Regular Air,Small Business,Office Supplies,...,East,New York,Smithtown,11787,2015-05-28,2015-05-28,261.8757,10,379.53,86839
3,26267,High,0.04,2.98,1.58,16,Sarah Ramsey,Regular Air,Small Business,Office Supplies,...,East,New York,Syracuse,13210,2015-02-12,2015-02-15,2.6300,6,18.80,86836
4,26268,High,0.05,115.99,2.50,16,Sarah Ramsey,Regular Air,Small Business,Technology,...,East,New York,Syracuse,13210,2015-02-12,2015-02-14,652.7331,10,945.99,86836


In [5]:
orders_clean = clean_orders(orders_raw, returns_raw)

Pocinje ciscenje podataka...

TRIM primenjen na sve string kolone
Postal Code formatiran na 5 cifara
Zaokruzene kolone: ['Discount', 'Unit Price', 'Shipping Cost', 'Product Base Margin', 'Profit', 'Sales']
Uklonjeno duplikata: 0 (bilo 1951, ostalo 1951)
Datumi konvertovani u datetime format
Dodata kolona: Days Between Order and Ship Date
Dodata kolona: is_returned (435 vracenih narudzbina)
Pronadjene null vrednosti:
  - Product Base Margin: 16 null vrednosti
  Product Base Margin: null zamenjeni medijanom (0.53)
Ukupno redova sa null vrednostima: 16

Ciscenje zavrseno! Finalni DataFrame: 1951 redova, 27 kolona


In [6]:
orders_clean.head()

,Row ID,Order Priority,Discount,Unit Price,Shipping Cost,Customer ID,Customer Name,Ship Mode,Customer Segment,Product Category,...,City,Postal Code,Order Date,Ship Date,Profit,Quantity ordered new,Sales,Order ID,Days Between Order and Ship Date,is_returned
0,21776,Critical,0.06,9.48,7.29,11,Marcus Dunlap,Regular Air,Home Office,Furniture,...,Roselle,07203,2015-02-15,2015-02-17,-53.81,22,211.15,90192,2,False
1,18181,Critical,0.00,4.42,4.99,15,Timothy Reese,Regular Air,Small Business,Office Supplies,...,Smithtown,11787,2015-04-08,2015-04-09,-59.82,7,33.47,86837,1,False
2,20925,Medium,0.01,35.94,6.66,15,Timothy Reese,Regular Air,Small Business,Office Supplies,...,Smithtown,11787,2015-05-28,2015-05-28,261.88,10,379.53,86839,0,False
3,26267,High,0.04,2.98,1.58,16,Sarah Ramsey,Regular Air,Small Business,Office Supplies,...,Syracuse,13210,2015-02-12,2015-02-15,2.63,6,18.80,86836,3,False
4,26268,High,0.05,115.99,2.50,16,Sarah Ramsey,Regular Air,Small Business,Technology,...,Syracuse,13210,2015-02-12,2015-02-14,652.73,10,945.99,86836,2,False


In [7]:
export_csv(orders_clean, PROCESSED_DIR + 'orders_clean.csv')
export_csv(returns_raw,  PROCESSED_DIR + 'returns_clean.csv')
export_csv(users_raw,    PROCESSED_DIR + 'users_clean.csv')

✅ Exportovano: data/processed/orders_clean.csv (1951 redova)
✅ Exportovano: data/processed/returns_clean.csv (300 redova)
✅ Exportovano: data/processed/users_clean.csv (4 redova)


In [8]:
# Upload orders u staging tabelu
print("Upload staging_orders...")
orders_clean.to_sql(
    name='staging_orders',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=500
)
print(f"staging_orders uploadovan: {len(orders_clean)} redova")

# Upload returns
print("\n Upload staging_returns...")
returns_raw.to_sql(
    name='staging_returns',
    con=engine,
    if_exists='replace',
    index=False
)
print(f" staging_returns uploadovan: {len(returns_raw)} redova")

# Upload users
print("\n Upload staging_users...")
users_raw.to_sql(
    name='staging_users',
    con=engine,
    if_exists='replace',
    index=False
)
print(f" staging_users uploadovan: {len(users_raw)} redova")

Upload staging_orders...
staging_orders uploadovan: 1951 redova

 Upload staging_returns...
 staging_returns uploadovan: 300 redova

 Upload staging_users...
 staging_users uploadovan: 4 redova


In [9]:
with engine.connect() as conn:

    # Punjenje Dim_Calendar sa Order Date i Ship Date
    conn.execute(text("""
        INSERT IGNORE INTO dim_calendar (date_id, full_date, year, quarter,
                                         month_number, month_name, week_number)
        SELECT DISTINCT
            CAST(DATE_FORMAT(`Order Date`, '%Y%m%d') AS UNSIGNED),
            DATE(`Order Date`),
            YEAR(`Order Date`),
            QUARTER(`Order Date`),
            MONTH(`Order Date`),
            MONTHNAME(`Order Date`),
            WEEK(`Order Date`)
        FROM staging_orders
        UNION
        SELECT DISTINCT
            CAST(DATE_FORMAT(`Ship Date`, '%Y%m%d') AS UNSIGNED),
            DATE(`Ship Date`),
            YEAR(`Ship Date`),
            QUARTER(`Ship Date`),
            MONTH(`Ship Date`),
            MONTHNAME(`Ship Date`),
            WEEK(`Ship Date`)
        FROM staging_orders;
    """))
    conn.commit()
    print("Dim_Calendar napunjena")

    # Punjenje Dim_Customers
    conn.execute(text("""
        INSERT IGNORE INTO dim_customers (customer_id, customer_name)
        SELECT DISTINCT `Customer ID`, `Customer Name`
        FROM staging_orders;
    """))
    conn.commit()
    print("Dim_Customers napunjena")

    # Punjenje Dim_Product - koristi AVG za nestabilan Product_Base_Margin
    conn.execute(text("""
        INSERT IGNORE INTO dim_product
            (product_name, product_category, product_sub_category,
             product_container, product_base_margin)
        SELECT
            `Product Name`,
            `Product Category`,
            `Product Sub-Category`,
            `Product Container`,
            ROUND(AVG(`Product Base Margin`), 2)
        FROM staging_orders
        GROUP BY `Product Name`, `Product Category`,
                 `Product Sub-Category`, `Product Container`;
    """))
    conn.commit()
    print("Dim_Product napunjena")

    # Punjenje Dim_Geography
    conn.execute(text("""
        INSERT IGNORE INTO dim_geography
            (country, region, state_or_province, city, postal_code)
        SELECT DISTINCT
            Country, Region, `State or Province`, City, `Postal Code`
        FROM staging_orders;
    """))
    conn.commit()
    print("Dim_Geography napunjena")

    # Punjenje Dim_Segment
    conn.execute(text("""
        INSERT IGNORE INTO dim_segment (segment_name)
        SELECT DISTINCT `Customer Segment`
        FROM staging_orders;
    """))
    conn.commit()
    print("Dim_Segment napunjena")

    # Punjenje Dim_ShipMod

Dim_Calendar napunjena
Dim_Customers napunjena
Dim_Product napunjena
Dim_Geography napunjena
Dim_Segment napunjena


In [10]:
# Provera da li sve dimenzije imaju podatke
with engine.connect() as conn:
    tables = [
        'dim_customers', 'dim_product', 'dim_geography',
        'dim_segment', 'dim_orderpriority', 'dim_shipmode',
        'dim_manager', 'dim_calendar'
    ]
    
    print("Broj redova po dimenziji:")
    print("-" * 35)
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.scalar()
        print(f"  {table:<25} {count:>6} redova")

Broj redova po dimenziji:
-----------------------------------
  dim_customers               1130 redova
  dim_product                  912 redova
  dim_geography               2955 redova
  dim_segment                    4 redova
  dim_orderpriority              5 redova
  dim_shipmode                   3 redova
  dim_manager                    8 redova
  dim_calendar                 188 redova


In [14]:
# Finalna verifikacija
with engine.connect() as conn:
    tables = [
        'fact_orders', 'dim_calendar', 'dim_customers',
        'dim_product', 'dim_geography', 'dim_manager',
        'dim_segment', 'dim_shipmode', 'dim_orderpriority'
    ]
    
    print("Finalni broj redova po tabeli:")
    print("-" * 35)
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.scalar()
        print(f"  {table:<25} {count:>6} redova")

Finalni broj redova po tabeli:
-----------------------------------
  fact_orders                 1951 redova
  dim_calendar                 188 redova
  dim_customers               1130 redova
  dim_product                  912 redova
  dim_geography               2955 redova
  dim_manager                    8 redova
  dim_segment                    4 redova
  dim_shipmode                   3 redova
  dim_orderpriority              5 redova


In [15]:
# Brzi test - pogledaj 5 redova sa svim dimenzijama
query = """
    SELECT
        f.row_id,
        c.customer_name,
        p.product_name,
        p.product_category,
        g.city,
        g.state_or_province,
        cal.full_date    AS order_date,
        m.manager_name,
        f.sales,
        f.profit,
        f.is_returned
    FROM fact_orders f
    JOIN dim_customers     c   ON c.customer_key  = f.customer_id
    JOIN dim_product       p   ON p.product_key   = f.product_id
    JOIN dim_geography     g   ON g.geography_id  = f.geography_id
    JOIN dim_calendar      cal ON cal.date_id     = f.order_date_id
    JOIN dim_manager       m   ON m.manager_id    = f.manager_id
    LIMIT 5;
"""

pd.read_sql(query, engine)

,row_id,customer_name,product_name,product_category,city,state_or_province,order_date,manager_name,sales,profit,is_returned
0,21776,Marcus Dunlap,"DAX Two-Tone Rosewood/Black Document Frame, De...",Furniture,Roselle,New Jersey,2015-02-15,Erin,211.15,-53.81,0
1,22293,Arthur Gold,"DAX Two-Tone Rosewood/Black Document Frame, De...",Furniture,Hendersonville,Tennessee,2015-01-14,Sam,12.90,238.93,0
2,23313,Laurence Cummings,"DAX Two-Tone Rosewood/Black Document Frame, De...",Furniture,Lehigh Acres,Florida,2015-03-24,Sam,20.22,-50.40,0
3,18181,Timothy Reese,Grip Seal Envelopes,Office Supplies,Smithtown,New York,2015-04-08,Erin,33.47,-59.82,0
4,20392,Judy Frazier,Grip Seal Envelopes,Office Supplies,East Massapequa,New York,2015-06-03,Erin,14.85,-10.44,0
